# Model Demo

# 1. Imports

In [31]:
import ipywidgets as widgets
from IPython.display import display
import joblib
import pandas as pd
import numpy as np

# 2. Load Model

In [32]:
rfc_loaded = joblib.load('../data/models/rfc_model.joblib')

In [33]:
model_columns = rfc_loaded.feature_names_in_

In [34]:
class_labels = ['Up to 1 day', '2 to 3 days', '4 to 7 days', '1 to 4 weeks', 'Over 1 month']

# 3. Predictor Function

In [35]:
def prediction_generator(topic, municipality, medium):
    user_input_topic = 'tema_solicitud_compact_' + topic
    user_input_municipality = 'alcaldia_solicitud_' + municipality
    user_input_input = 'tipo_de_entrada_' + medium
    model_df = pd.DataFrame(data=np.zeros(68), index=model_columns).transpose()
    model_df[user_input_topic] = 1
    model_df[user_input_municipality] = 1
    model_df[user_input_input] = 1
    y_pred = rfc_loaded.predict_proba(model_df)
    pred_results = pd.Series(y_pred[0], index=rfc_loaded.classes_).reindex(index=class_labels)
    return pred_results


In [36]:
prediction = prediction_generator('BACHEO','Gustavo A. Madero','Portal Ciudadano SUAC')
prediction

Up to 1 day     0.063025
2 to 3 days     0.079852
4 to 7 days     0.093505
1 to 4 weeks    0.188924
Over 1 month    0.574695
dtype: float64

In [37]:
topic_options = []
municipality_options = []
medium_options = []

for column in model_columns:
    if column.startswith('tema_solicitud_compact_'):
        option = column.removeprefix('tema_solicitud_compact_')
        topic_options.append(option)
    elif column.startswith('alcaldia_solicitud_'):
        option = column.removeprefix('alcaldia_solicitud_')
        municipality_options.append(option)
    elif column.startswith('tipo_de_entrada_'):
        option = column.removeprefix('tipo_de_entrada_')
        medium_options.append(option)

topic_options.sort()
municipality_options.sort()
medium_options.sort()

# 4. Widget Definition

In [38]:
dropdown_topic = widgets.Dropdown(
    options=topic_options,
    description = 'Topic',
    style = {'description_width' : '120px'}
)

In [39]:
dropdown_municipality = widgets.Dropdown(
    options=municipality_options,
    description = 'Municipality',
    style = {'description_width' : '120px'}
)

In [40]:
dropdown_medium = widgets.Dropdown(
    options=medium_options,
    description = 'Input Medium',
    style = {'description_width' : '120px'}
)

In [ ]:
button = widgets.Button(
    description = 'Predict Time',
    button_style = 'primary'
)

In [42]:
output = widgets.Output()

# 5. Interaction

In [43]:
def interactive_func(b):
    with output:
        output.clear_output(wait=True)
        topic_selection = dropdown_topic.value
        municipality_selection = dropdown_municipality.value
        input_selection = dropdown_medium.value
        pred = prediction_generator(topic_selection, municipality_selection, input_selection)
        most_prob_class = pred.idxmax()
        most_prob_value = pred.max()
        print(f'Most probable class: {most_prob_class}')
        print(f'Most probable class value: {most_prob_value*100:.1f}%')
        display(pred)

In [44]:
button.on_click(interactive_func)

In [45]:
display(widgets.VBox([dropdown_topic, dropdown_municipality, dropdown_medium, button, output]))